# 02 · Tool, filesystem ed esecuzione

Gli agenti diventano utili quando **agiscono** sul mondo: leggono e scrivono file, eseguono
codice. Qui vediamo tre cose, con attenzione alla **sicurezza**:
1. un tool tipizzato ben definito;
2. tool su file **confinati** in una cartella (niente path traversal);
3. eseguire codice in un **sottoprocesso** isolato con timeout.

## Setup (autonomo)

Ogni notebook è **indipendente**: non importa nulla dal progetto. Qui carichiamo la chiave
API dal file `.env` e creiamo un modello. Esegui le celle in ordine dall'alto verso il basso.

In [ ]:
# Carichiamo le variabili d'ambiente dal file `.env`.
# Lo cerchiamo nella cartella corrente e in quelle superiori, così il notebook
# funziona sia se avviato dalla radice del progetto sia dalla cartella `notebooks`.
import os
from pathlib import Path

from dotenv import load_dotenv


def trova_env() -> Path:
    for cartella in (Path.cwd(), *Path.cwd().resolve().parents):
        if (cartella / ".env").is_file():
            return cartella / ".env"
    raise FileNotFoundError("File .env non trovato: copia .env.example in .env e aggiungi la chiave.")


env_file = trova_env()
load_dotenv(env_file, override=False)          # carica le variabili senza sovrascrivere quelle già presenti
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY mancante nel file .env"
print("Ambiente caricato da:", env_file)

In [ ]:
# `ChatOpenAI` è il wrapper LangChain attorno al modello.
# Lo creiamo una volta e lo riusiamo in tutto il notebook.
from langchain_openai import ChatOpenAI

MODELLO = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")   # modello economico, va bene per imparare
model = ChatOpenAI(
    model=MODELLO,
    use_responses_api=True,   # API "responses" di OpenAI
    store=False,              # non conservare la conversazione sui server OpenAI
)
print("Modello pronto:", MODELLO)

## 1 · Un tool tipizzato

Il modello sceglie un tool leggendo nome, tipi e docstring. Più sono chiari, meglio sceglie.

In [ ]:
from langchain_core.tools import tool


@tool
def conta_parole(testo: str) -> int:
    """Conta quante parole ci sono in un testo."""
    return len(testo.split())


# Possiamo provare un tool anche senza agente, con `.invoke`.
print(conta_parole.invoke({"testo": "uno due tre"}))

## 2 · File confinati in una cartella (workspace)

Un agente che scrive file è potente ma pericoloso. La difesa: consentire **solo** una
cartella "workspace" e rifiutare qualsiasi percorso che ne esca (es. `../../etc/passwd`).

In [ ]:
import tempfile
from pathlib import Path

# Creiamo una cartella temporanea che fa da "workspace" sicuro.
WORKSPACE = Path(tempfile.mkdtemp(prefix="nb_workspace_"))
print("Workspace:", WORKSPACE)


def percorso_sicuro(nome: str) -> Path:
    # `resolve()` normalizza il path (risolve i `..`); poi controlliamo che resti DENTRO il workspace.
    candidato = (WORKSPACE / nome).resolve()
    if not candidato.is_relative_to(WORKSPACE.resolve()):
        raise ValueError(f"Percorso fuori dal workspace: {nome}")
    return candidato

In [ ]:
# Verifica veloce della difesa: un percorso malevolo viene bloccato.
try:
    percorso_sicuro("../fuori.txt")
except ValueError as errore:
    print("Bloccato correttamente:", errore)

In [ ]:
# Ora i due tool su file: entrambi passano da `percorso_sicuro`.
@tool
def scrivi_file(nome: str, contenuto: str) -> str:
    """Scrive un file di testo nel workspace."""
    percorso = percorso_sicuro(nome)
    percorso.write_text(contenuto, encoding="utf-8")
    return f"Scritti {len(contenuto)} caratteri in {nome}"


@tool
def leggi_file(nome: str) -> str:
    """Legge un file di testo dal workspace."""
    return percorso_sicuro(nome).read_text(encoding="utf-8")

## 3 · Eseguire codice in un sottoprocesso isolato

Per far "provare" del codice all'agente lo eseguiamo in un **processo separato**, con un
**timeout**. In produzione si usa un container (es. Docker); qui basta il concetto: l'output
(stdout, stderr, exit code) torna come *osservazione*, non blocca mai il programma.

In [ ]:
import subprocess


@tool
def esegui_python(codice: str) -> str:
    """Esegue un piccolo script Python e restituisce l'output."""
    try:
        risultato = subprocess.run(
            ["python", "-c", codice],   # niente shell: passiamo gli argomenti come lista
            capture_output=True, text=True, timeout=10, check=False,
        )
    except subprocess.TimeoutExpired:
        return "ERRORE: esecuzione troppo lunga (timeout)."
    # Restituiamo tutto: l'exit code dice se è andata bene (0 = ok).
    return f"exit={risultato.returncode}\nSTDOUT:\n{risultato.stdout}\nSTDERR:\n{risultato.stderr}"

## 4 · Un agente che usa i tool

Diamo tutti i tool all'agente e chiediamo un compito che richiede di scrivere, eseguire e verificare.

In [ ]:
from langchain.agents import create_agent

agente = create_agent(
    model=model,
    tools=[conta_parole, scrivi_file, leggi_file, esegui_python],
    system_prompt=(
        "Puoi scrivere file nel workspace ed eseguire Python per verificare. "
        "Dopo aver scritto codice, eseguilo e controlla l'exit code prima di concludere."
    ),
)

In [ ]:
esito = agente.invoke({"messages": [{
    "role": "user",
    "content": "Scrivi somma.py che stampa la somma da 1 a 10, poi eseguilo e dimmi il risultato.",
}]})
print(esito["messages"][-1].text)

## Prova tu

- Chiedi all'agente di leggere un file inesistente: vedrai l'errore tornare come osservazione.
- Aggiungi un tool `elenca_file` che lista il workspace.

**Idea chiave**: un tool è un confine di fiducia. Validare i percorsi e isolare l'esecuzione
è ciò che rende sicuro dare "mani" a un agente.